# Vacuum Table Identification - Hive Metastore Analysis

This notebook identifies Delta tables in Hive Metastore that have multiple versions and may need vacuuming.

## Overview
- Traverses all databases and tables in HMS
- Identifies Delta tables with version history
- Extracts metadata from DESCRIBE HISTORY
- Collects operational metrics including row counts
- Provides recommendations for vacuum operations


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import json
from datetime import datetime

# Initialize Spark session (already available in Databricks)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

print("Libraries imported successfully!")
print(f"Analysis started at: {datetime.now()}")


Libraries imported successfully!
Analysis started at: 2026-02-17 18:45:32.842060


## Configuration Parameters - Create Widgets


In [0]:
# Create Databricks widgets for configuration
dbutils.widgets.text("catalog", "hive_metastore", "1. Catalog Name")
dbutils.widgets.text("target_databases", "ade", "2. Target Databases (comma-separated, or leave empty for all)")
dbutils.widgets.text("exclude_databases", "information_schema,sys,default", "3. Exclude Databases (comma-separated)")
dbutils.widgets.text("min_versions_threshold", "5", "4. Minimum Versions for Vacuum Recommendation")

# Storage filtering widget
dbutils.widgets.dropdown("filter_managed_only", "Yes", ["Yes", "No"], "5. Filter Managed Storage Only")

# Output configuration widgets
dbutils.widgets.text("output_database", "default", "6. Output Database (for saving results)")
dbutils.widgets.text("output_table", "vacuum_analysis_results", "7. Output Table Name")

print("✓ Widgets created! Use the widget controls at the top to configure the analysis.")


✓ Widgets created! Use the widget controls at the top to configure the analysis.


## Step 1: Read Configuration from Widgets


In [0]:
# Read configuration from widgets
CATALOG = dbutils.widgets.get("catalog").strip()

# Parse target databases (comma-separated)
target_databases_str = dbutils.widgets.get("target_databases").strip()
if target_databases_str:
    TARGET_DATABASES = [db.strip() for db in target_databases_str.split(",") if db.strip()]
else:
    TARGET_DATABASES = []

# Parse exclude databases (comma-separated)
exclude_databases_str = dbutils.widgets.get("exclude_databases").strip()
EXCLUDE_DATABASES = [db.strip() for db in exclude_databases_str.split(",") if db.strip()]

# Parse minimum versions threshold
MIN_VERSIONS_FOR_VACUUM = int(dbutils.widgets.get("min_versions_threshold"))

# Parse storage filter configuration
FILTER_MANAGED_ONLY = dbutils.widgets.get("filter_managed_only").strip().lower() == "yes"

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Catalog: {CATALOG}")
print(f"  ⚠️  NOTE: Analysis will ONLY scan databases within '{CATALOG}' catalog")
print(f"  Other Unity Catalog catalogs will be ignored")
print()
print(f"Target Databases: {'All databases in ' + CATALOG if not TARGET_DATABASES else ', '.join(TARGET_DATABASES)}")
print(f"Exclude Databases: {', '.join(EXCLUDE_DATABASES) if EXCLUDE_DATABASES else 'None'}")
print(f"Min Versions Threshold: {MIN_VERSIONS_FOR_VACUUM}")
print(f"Filter Managed Storage Only: {'Yes' if FILTER_MANAGED_ONLY else 'No'}")
if FILTER_MANAGED_ONLY:
    print(f"  ℹ️  Only tables stored in Databricks managed storage will be included")
else:
    print(f"  ℹ️  All tables will be included (managed and external storage)")
print("="*80)


CONFIGURATION
Catalog: hive_metastore
  ⚠️  NOTE: Analysis will ONLY scan databases within 'hive_metastore' catalog
  Other Unity Catalog catalogs will be ignored

Target Databases: All databases in hive_metastore
Exclude Databases: information_schema, sys
Min Versions Threshold: 2
Filter Managed Storage Only: Yes
  ℹ️  Only tables stored in Databricks managed storage will be included


## Step 2: Get All Databases from HMS


In [0]:
# Get all databases from the CATALOG ONLY (not from other Unity Catalogs)
print(f"Fetching databases from catalog: {CATALOG}")

try:
    # Use catalog-specific query to ensure we only get databases from hive_metastore
    databases_df = spark.sql(f"SHOW DATABASES IN {CATALOG}")
    print(f"✓ Successfully connected to catalog: {CATALOG}")
except Exception as e:
    print(f"✗ Error: Could not access catalog '{CATALOG}': {str(e)}")
    print(f"  Make sure the catalog exists and you have access")
    raise

# Filter databases based on TARGET_DATABASES
if TARGET_DATABASES:
    # User specified specific databases - use them
    databases = TARGET_DATABASES
    print(f"Using specified databases: {', '.join(databases)}")
else:
    # Get all databases from the catalog, excluding system databases
    all_databases = [row.databaseName for row in databases_df.collect()]
    databases = [db for db in all_databases if db not in EXCLUDE_DATABASES]
    print(f"Scanning ALL databases in {CATALOG} (excluding system databases)")

# Verify all specified databases exist in the catalog
available_databases = {row.databaseName for row in databases_df.collect()}
missing_databases = [db for db in databases if db not in available_databases]

if missing_databases:
    print(f"\n⚠️  WARNING: The following databases were not found in {CATALOG}:")
    for db in missing_databases:
        print(f"    - {db}")
    # Remove missing databases from the list
    databases = [db for db in databases if db in available_databases]
    print(f"\nProceeding with {len(databases)} valid databases")

print(f"\n{'='*80}")
print(f"DATABASES TO SCAN IN {CATALOG}")
print(f"{'='*80}")
print(f"Total databases: {len(databases)}")
for i, db in enumerate(databases, 1):
    print(f"  {i}. {db}")
print(f"{'='*80}\n")


Fetching databases from catalog: hive_metastore
✓ Successfully connected to catalog: hive_metastore
Scanning ALL databases in hive_metastore (excluding system databases)

DATABASES TO SCAN IN hive_metastore
Total databases: 47
  1. _default_location
  2. ali_d
  3. alid_test
  4. andrew_coannt
  5. aspanalytics
  6. baiyi_zhang
  7. base
  8. bsa_analytics_reporting_dbr
  9. business_solutions
  10. business_unit_reporting
  11. cashlessbranches
  12. cc_acq
  13. cc_bop
  14. ccobids
  15. ccobids_pbi_base
  16. ccobids_pbi_scorecard
  17. curated
  18. default
  19. dmca
  20. eats_pipeline
  21. emessage
  22. file_sla
  23. hunterh
  24. internal_chat_ach_message_records
  25. internal_chat_int_ach_receipts
  26. internal_chat_mobile_atm
  27. internal_chat_mobile_atm_history
  28. internal_chat_mobile_atm_message_records
  29. jenn_song
  30. lorena_guimaraes
  31. mangold
  32. md_mgr_mbr
  33. mina_yen
  34. my_hive_db
  35. nan_li
  36. rahil_pereira
  37. rahul
  38. reporting

## Step 3: Helper Functions


In [0]:
def is_delta_table(database, table):
    """Check if a table is a Delta table"""
    try:
        # Use 3-level naming with catalog
        full_table_name = f"`{CATALOG}`.`{database}`.`{table}`"
        table_details = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()
        if table_details and table_details[0].format == 'delta':
            return True, table_details[0]
        return False, None
    except Exception as e:
        # Try 2-level naming as fallback
        try:
            table_details = spark.sql(f"DESCRIBE DETAIL `{database}`.`{table}`").collect()
            if table_details and table_details[0].format == 'delta':
                return True, table_details[0]
        except:
            pass
        return False, None

def get_table_history(database, table):
    """Get history information for a Delta table"""
    try:
        # Use 3-level naming with catalog
        full_table_name = f"`{CATALOG}`.`{database}`.`{table}`"
        history_df = spark.sql(f"DESCRIBE HISTORY {full_table_name}")
        return history_df
    except Exception as e:
        # Try 2-level naming as fallback
        try:
            history_df = spark.sql(f"DESCRIBE HISTORY `{database}`.`{table}`")
            return history_df
        except:
            print(f"Error getting history for {database}.{table}: {str(e)}")
            return None

def extract_row_count_from_metrics(operation_metrics):
    """Extract row count from operation metrics"""
    try:
        if operation_metrics:
            # Handle both dict and string formats
            metrics = operation_metrics if isinstance(operation_metrics, dict) else {}
            # Try different metric names
            for key in ['numOutputRows', 'numRecords', 'numRows', 'numTargetRowsInserted', 
                       'numTargetRowsUpdated', 'numTargetRowsCopied']:
                if key in metrics:
                    return int(metrics[key])
        return None
    except:
        return None

def extract_files_added_removed(operation_metrics):
    """Extract files added/removed from operation metrics"""
    try:
        if operation_metrics:
            # Handle both dict and string formats
            metrics = operation_metrics if isinstance(operation_metrics, dict) else {}
            files_added = int(metrics.get('numAddedFiles', 0))
            files_removed = int(metrics.get('numRemovedFiles', 0))
            return files_added, files_removed
        return 0, 0
    except:
        return 0, 0
def is_managed_storage(location):

    if not location:
        return False, "Unknown", False
    
    location_lower = location.lower()
    
    # Check for Azure mount points (dbfs:/mnt/)
    azure_mount_patterns = [
        'dbfs:/mnt/',
        '/dbfs/mnt/',
    ]
    
    is_azure_mount = any(pattern in location_lower for pattern in azure_mount_patterns)
    
    if is_azure_mount:
        # Azure mount points are treated as external storage
        return False, "Azure Mount Point", True
    
    # Check for external storage patterns (Azure, AWS, GCP)
    external_patterns = [
        'abfss://',      # Azure Data Lake Storage Gen2
        'wasbs://',      # Azure Blob Storage
        'wasb://'       # Azure Blob Storage

    ]
    
    for pattern in external_patterns:
        if pattern in location_lower:
            # Additional check: if it contains storage account name, it's definitely external
            if '@' in location_lower or 'storageaccount' in location_lower:
                return False, "External", False
            # For s3/gs, they're always external
            if pattern in ['s3://', 's3a://', 's3n://', 'gs://']:
                return False, "External", False
            return False, "External", False
    
    # Check for managed storage patterns
    managed_patterns = [
        'dbfs:/user/hive/warehouse/',
        '/dbfs/user/hive/warehouse/',
        'dbfs:/databricks/',
        '/dbfs/databricks/',
    ]
    
    for pattern in managed_patterns:
        if pattern in location_lower:
            return True, "Managed", False
    
    # If location starts with dbfs:/ but is not in external patterns or mount, likely managed
    if location_lower.startswith('dbfs:/') or location_lower.startswith('/dbfs/'):
        return True, "Managed", False
    
    # Default: if no explicit storage provider detected, consider it managed
    return True, "Likely Managed", False

def get_storage_type(location):
    """
    Get a human-readable storage type classification
    """
    is_managed, storage_type, is_mount = is_managed_storage(location)
    
    # If it's an Azure mount point, return specific classification
    if is_mount:
        return "Azure Mount Point"
    
    if is_managed:
        return "Managed"
    else:
        location_lower = location.lower() if location else ""
        if 'abfss://' in location_lower or 'wasbs://' in location_lower or 'wasb://' in location_lower:
            return "Azure External"
        elif 's3://' in location_lower or 's3a://' in location_lower or 's3n://' in location_lower:
            return "AWS S3"
        elif 'gs://' in location_lower:
            return "GCP GCS"
        else:
            return "External"

print("Storage detection functions defined successfully!")
print("  📌 Azure mount points (dbfs:/mnt/) are automatically detected and treated as external")


print("Helper functions defined successfully!")

table_analysis_results = []

# Counter for progress tracking
total_tables_scanned = 0
delta_tables_found = 0
tables_needing_vacuum = 0

print(f"Starting table scan within catalog: {CATALOG}")
print(f"⚠️  NOTE: Only scanning databases in '{CATALOG}' - other Unity Catalog catalogs will be ignored\n")

for database in databases:
    print(f"Scanning database: {CATALOG}.{database}")
    
    try:
        # Get all tables in the database from the specified CATALOG only
        # This ensures we only query tables within hive_metastore (or specified catalog)
        tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{database}")
        
        tables = [(row.database, row.tableName) for row in tables_df.collect()]
        
        print(f"  Found {len(tables)} tables in {database}")
        
        for db, table in tables:
            total_tables_scanned += 1
            
            # Check if it's a Delta table
            is_delta, table_detail = is_delta_table(db, table)
            
            if not is_delta:
                continue
            
            delta_tables_found += 1
            print(f"    Processing Delta table: {db}.{table}")
            
            # Get history
            history_df = get_table_history(db, table)
            
            if history_df is None:
                continue
            
            history_data = history_df.collect()
            version_count = len(history_data)
            
            # Only process tables with more than 1 version
            if version_count > 1:
                tables_needing_vacuum += 1
                
                try:
                    # Get latest version info
                    latest_version = history_data[0]
                    
                    # Get oldest version info for time span calculation
                    oldest_version = history_data[-1]
                    
                    # Extract row count from latest version's operational metrics
                    latest_row_count = extract_row_count_from_metrics(
                        latest_version.operationMetrics if hasattr(latest_version, 'operationMetrics') else None
                    )
                    
                    # Calculate total files added and removed across all versions
                    total_files_added = 0
                    total_files_removed = 0
                    operation_types = {}
                    
                    for version in history_data:
                        # Count operations
                        op = version.operation
                        operation_types[op] = operation_types.get(op, 0) + 1
                        
                        # Count files
                        if hasattr(version, 'operationMetrics') and version.operationMetrics:
                            files_added, files_removed = extract_files_added_removed(version.operationMetrics)
                            total_files_added += files_added
                            total_files_removed += files_removed
                    
                    # Calculate potential files to vacuum (files that were removed/replaced)
                    potential_vacuum_files = total_files_removed
                    
                    # Get table size
                    table_size_bytes = table_detail.sizeInBytes if hasattr(table_detail, 'sizeInBytes') else None
                    table_location = table_detail.location if hasattr(table_detail, 'location') else None
                    
                    # Calculate days since last modification
                    from datetime import datetime, timezone
                    if hasattr(latest_version, 'timestamp'):
                        last_modified = latest_version.timestamp
                        if last_modified.tzinfo is None:
                            last_modified = last_modified.replace(tzinfo=timezone.utc)
                        days_since_modification = (datetime.now(timezone.utc) - last_modified).days
                    else:
                        days_since_modification = None
                    
                    # Calculate version span in days
                    if hasattr(latest_version, 'timestamp') and hasattr(oldest_version, 'timestamp'):
                        version_span_days = (latest_version.timestamp - oldest_version.timestamp).days
                    else:
                        version_span_days = None
                    
                    # Determine vacuum priority
                    vacuum_priority = "LOW"
                    if version_count >= MIN_VERSIONS_FOR_VACUUM:
                        if potential_vacuum_files > 100:
                            vacuum_priority = "HIGH"
                        elif potential_vacuum_files > 50:
                            vacuum_priority = "MEDIUM"
                    
                    # Calculate table size in MB (avoid PySpark round() conflict)
                    if table_size_bytes:
                        table_size_mb = float(table_size_bytes) / (1024*1024)
                        table_size_mb = float(f"{table_size_mb:.2f}")  # Round to 2 decimal places
                    else:
                        table_size_mb = None
                    
                    # Determine storage type
                    # Determine storage type with mount point treatment
                    is_managed, _, is_mount = is_managed_storage(table_location)
                    storage_type = get_storage_type(table_location)
                    
                    # Apply storage filter if enabled
                    if FILTER_MANAGED_ONLY and not is_managed:
                        print(f"    ⊗ Skipping (external storage): {db}.{table} - {storage_type}")
                        continue
                    
                    # Store results
                    table_info = {
                        'database': db,
                        'table': table,
                        'table_location': table_location,
                        'storage_type': storage_type,
                        'is_managed_storage': is_managed,
                        'is_mount_point': is_mount,
                        'version_count': version_count,
                        'latest_version': latest_version.version if hasattr(latest_version, 'version') else None,
                        'latest_operation': latest_version.operation if hasattr(latest_version, 'operation') else None,
                        'last_modified_by': latest_version.userName if hasattr(latest_version, 'userName') else None,
                        'last_modified_timestamp': latest_version.timestamp if hasattr(latest_version, 'timestamp') else None,
                        'days_since_modification': days_since_modification,
                        'version_span_days': version_span_days,
                        'oldest_version_timestamp': oldest_version.timestamp if hasattr(oldest_version, 'timestamp') else None,
                        'latest_row_count': latest_row_count,
                        'table_size_bytes': table_size_bytes,
                        'table_size_mb': table_size_mb,
                        'total_files_added': total_files_added,
                        'total_files_removed': total_files_removed,
                        'potential_vacuum_files': potential_vacuum_files,
                        'operation_summary': str(operation_types),
                        'vacuum_priority': vacuum_priority
                    }
                    
                    table_analysis_results.append(table_info)
                    print(f"    ✓ Added to results: {version_count} versions, {potential_vacuum_files} files to vacuum, Storage: {storage_type}")
                    
                except Exception as e:
                    print(f"    ✗ Error processing table data: {str(e)}")
                    import traceback
                    print(f"    Details: {traceback.format_exc()}")
                
    except Exception as e:
        print(f"  Error scanning database {database}: {str(e)}")
        continue

print(f"\n{'='*80}")
print(f"Scan completed for catalog: {CATALOG}")
print(f"{'='*80}")
print(f"Total tables scanned: {total_tables_scanned}")
print(f"Delta tables found: {delta_tables_found}")
print(f"Tables with multiple versions: {tables_needing_vacuum}")
print(f"\n⚠️  NOTE: Analysis was limited to '{CATALOG}' catalog only")
print(f"{'='*80}")


Storage detection functions defined successfully!
  📌 Azure mount points (dbfs:/mnt/) are automatically detected and treated as external
Helper functions defined successfully!
Starting table scan within catalog: hive_metastore
⚠️  NOTE: Only scanning databases in 'hive_metastore' - other Unity Catalog catalogs will be ignored

Scanning database: hive_metastore._default_location
  Found 0 tables in _default_location
Scanning database: hive_metastore.ali_d
  Found 1 tables in ali_d
    Processing Delta table: ali_d.final_result
Scanning database: hive_metastore.alid_test
  Found 0 tables in alid_test
Scanning database: hive_metastore.andrew_coannt
  Found 0 tables in andrew_coannt
Scanning database: hive_metastore.aspanalytics
  Found 3 tables in aspanalytics
    Processing Delta table: aspanalytics.alert_usage_metrics
    ✓ Added to results: 40 versions, 29 files to vacuum, Storage: Managed
    Processing Delta table: aspanalytics.alert_usage_metrics_by_action
    ✓ Added to results: 13

## Step 4: Scan All Tables and Collect Metadata

**Important**: This step scans ONLY tables within the specified catalog (e.g., `hive_metastore`).  
Tables from other Unity Catalog catalogs will not be included in the analysis.


In [0]:
# Convert results to Spark DataFrame
if table_analysis_results:
    # Define explicit schema for Spark Connect compatibility
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType, BooleanType
    
    schema = StructType([
        StructField("database", StringType(), True),
        StructField("table", StringType(), True),
        StructField("table_location", StringType(), True),
        StructField("storage_type", StringType(), True),
        StructField("is_managed_storage", BooleanType(), True),
        StructField("is_mount_point", BooleanType(), True),
        StructField("version_count", IntegerType(), True),
        StructField("latest_version", LongType(), True),
        StructField("latest_operation", StringType(), True),
        StructField("last_modified_by", StringType(), True),
        StructField("last_modified_timestamp", TimestampType(), True),
        StructField("days_since_modification", IntegerType(), True),
        StructField("version_span_days", IntegerType(), True),
        StructField("oldest_version_timestamp", TimestampType(), True),
        StructField("latest_row_count", LongType(), True),
        StructField("table_size_bytes", LongType(), True),
        StructField("table_size_mb", DoubleType(), True),
        StructField("total_files_added", IntegerType(), True),
        StructField("total_files_removed", IntegerType(), True),
        StructField("potential_vacuum_files", IntegerType(), True),
        StructField("operation_summary", StringType(), True),
        StructField("vacuum_priority", StringType(), True)
    ])
    
    results_df = spark.createDataFrame(table_analysis_results, schema=schema)
    
    # Sort by vacuum priority and version count
    results_df = results_df.orderBy(
        when(col('vacuum_priority') == 'HIGH', 1)
        .when(col('vacuum_priority') == 'MEDIUM', 2)
        .otherwise(3),
        col('version_count').desc()
    )
    
    print(f"Created DataFrame with {results_df.count()} tables needing attention")
else:
    print("No tables with multiple versions found")
    results_df = None


Created DataFrame with 159 tables needing attention


## Step 5: Convert Results to DataFrame for Analysis


In [0]:
if results_df:
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    
    # Vacuum priority distribution
    print("\nVacuum Priority Distribution:")
    results_df.groupBy('vacuum_priority').count().orderBy('vacuum_priority').show()
    
    # Top tables by version count
    print("\nTop 10 Tables by Version Count:")
    results_df.select(
        'database', 'table', 'version_count', 'potential_vacuum_files', 'vacuum_priority'
    ).orderBy(col('version_count').desc()).show(10, truncate=False)
    
    # Statistics
    print("\nOverall Statistics:")
    results_df.select(
        avg('version_count').alias('avg_versions'),
        max('version_count').alias('max_versions'),
        sum('potential_vacuum_files').alias('total_potential_vacuum_files'),
        sum('table_size_mb').alias('total_size_mb')
    ).show()



SUMMARY STATISTICS

Vacuum Priority Distribution:
+---------------+-----+
|vacuum_priority|count|
+---------------+-----+
|           HIGH|   15|
|            LOW|  144|
+---------------+-----+


Top 10 Tables by Version Count:
+---------------------+------------------------------+-------------+----------------------+---------------+
|database             |table                         |version_count|potential_vacuum_files|vacuum_priority|
+---------------------+------------------------------+-------------+----------------------+---------------+
|hunterh              |guardian_achr_payment_events  |389          |0                     |LOW            |
|eats_pipeline        |master_escheatment            |277          |24423                 |HIGH           |
|reporting            |blend_loans_bi                |78           |292486                |HIGH           |
|reporting            |zelle_pymts_alltransactions_bi|76           |0                     |LOW            |
|reporting     

## Step 6: Display Summary Statistics


In [0]:
if results_df:
    print("\nDETAILED RESULTS - All Tables with Multiple Versions")
    print("="*80)
    
    # Display all columns
    display(results_df)



DETAILED RESULTS - All Tables with Multiple Versions


database,table,table_location,storage_type,is_managed_storage,is_mount_point,version_count,latest_version,latest_operation,last_modified_by,last_modified_timestamp,days_since_modification,version_span_days,oldest_version_timestamp,latest_row_count,table_size_bytes,table_size_mb,total_files_added,total_files_removed,potential_vacuum_files,operation_summary,vacuum_priority
eats_pipeline,master_escheatment,dbfs:/user/hive/warehouse/eats_pipeline.db/master_escheatment,Managed,true,false,277,276,OPTIMIZE,john_mong@navyfederal.org,2026-02-17T15:55:42Z,0,196,2025-08-05T14:13:48Z,null,15409769703,14695.9,7666,24423,24423,"{'OPTIMIZE': 137, 'WRITE': 136, 'SET TBLPROPERTIES': 2, 'REPLACE TABLE AS SELECT': 1, 'CREATE TABLE AS SELECT': 1}",HIGH
reporting,blend_loans_bi,dbfs:/user/hive/warehouse/reporting.db/blend_loans_bi,Managed,true,false,78,996,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-17T00:12:38Z,0,42,2026-01-05T00:32:23Z,null,41366232196,39449.91,817,292486,292486,"{'VACUUM END': 4, 'VACUUM START': 5, 'CREATE OR REPLACE TABLE AS SELECT': 8, 'OPTIMIZE': 61}",HIGH
cc_acq,acctmob,dbfs:/user/hive/warehouse/cc_acq.db/acctmob,Managed,true,false,40,173,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:50:56Z,0,110,2025-10-29T09:53:46Z,null,4753340100,4533.14,314,5232,5232,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 17, 'CREATE OR REPLACE TABLE AS SELECT': 17}",HIGH
cc_acq,lao_tnl_pop,dbfs:/user/hive/warehouse/cc_acq.db/lao_tnl_pop,Managed,true,false,37,86,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:50:36Z,0,137,2025-10-02T20:05:56Z,null,4132877,3.94,6,4613,4613,"{'VACUUM END': 3, 'VACUUM START': 3, 'WRITE': 24, 'OPTIMIZE': 6, 'CREATE OR REPLACE TABLE AS SELECT': 1}",HIGH
cc_acq,acct_txn,dbfs:/user/hive/warehouse/cc_acq.db/acct_txn,Managed,true,false,30,43,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:50:15Z,0,542,2024-08-23T20:16:00Z,null,3061029784,2919.23,144,2597,2597,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 12, 'CREATE OR REPLACE TABLE AS SELECT': 12}",HIGH
cc_acq,acct_txn_stage_1,dbfs:/user/hive/warehouse/cc_acq.db/acct_txn_stage_1,Managed,true,false,20,19,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:49:44Z,0,525,2024-09-09T19:37:33Z,null,5000921676,4769.25,122,1042,1042,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 7, 'CREATE OR REPLACE TABLE AS SELECT': 7}",HIGH
taiwan_savage,propensity_threshold_backtest,dbfs:/user/hive/warehouse/taiwan_savage.db/propensity_threshold_backtest,Managed,true,false,11,10,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:49:02Z,0,136,2025-10-03T00:58:00Z,null,10681,0.01,2,222,222,"{'VACUUM END': 3, 'VACUUM START': 3, 'CREATE OR REPLACE TABLE AS SELECT': 3, 'OPTIMIZE': 2}",HIGH
taiwan_savage,threshold_impacts,dbfs:/user/hive/warehouse/taiwan_savage.db/threshold_impacts,Managed,true,false,9,8,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:48:35Z,0,138,2025-10-01T10:01:30Z,null,24928,0.02,1,111,111,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 1, 'CREATE OR REPLACE TABLE AS SELECT': 2}",HIGH
cc_acq,txnmob_tyler,dbfs:/user/hive/warehouse/cc_acq.db/txnmob_tyler,Managed,true,false,8,7,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:49:24Z,0,24,2026-01-23T22:56:02Z,null,3097048159,2953.58,13,512,512,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 1, 'CREATE OR REPLACE TABLE AS SELECT': 1}",HIGH
cc_acq,app,dbfs:/user/hive/warehouse/cc_acq.db/app,Managed,true,false,8,7,VACUUM END,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:48:43Z,0,377,2025-02-04T13:58:22Z,null,63977385,61.01,1,128,128,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 1, 'CREATE OR REPLACE TABLE AS SELECT': 1}",HIGH


## Step 7: Display Detailed Results


In [0]:
if results_df:
    high_priority_df = results_df.filter(col('vacuum_priority') == 'HIGH')
    
    if high_priority_df.count() > 0:
        print("\n" + "="*80)
        print("HIGH PRIORITY TABLES - Immediate Action Recommended")
        print("="*80)
        
        display(high_priority_df.select(
            'database',
            'table',
            'version_count',
            'last_modified_by',
            'last_modified_timestamp',
            'days_since_modification',
            'potential_vacuum_files',
            'table_size_mb',
            'operation_summary'
        ))
    else:
        print("\nNo high priority tables found")



HIGH PRIORITY TABLES - Immediate Action Recommended


database,table,version_count,last_modified_by,last_modified_timestamp,days_since_modification,potential_vacuum_files,table_size_mb,operation_summary
eats_pipeline,master_escheatment,277,john_mong@navyfederal.org,2026-02-17T15:55:42Z,0,24423,14695.9,"{'OPTIMIZE': 137, 'WRITE': 136, 'SET TBLPROPERTIES': 2, 'REPLACE TABLE AS SELECT': 1, 'CREATE TABLE AS SELECT': 1}"
reporting,blend_loans_bi,78,nayanjyoti_sonowal@navyfederal.org,2026-02-17T00:12:38Z,0,292486,39449.91,"{'VACUUM END': 4, 'VACUUM START': 5, 'CREATE OR REPLACE TABLE AS SELECT': 8, 'OPTIMIZE': 61}"
cc_acq,acctmob,40,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:50:56Z,0,5232,4533.14,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 17, 'CREATE OR REPLACE TABLE AS SELECT': 17}"
cc_acq,lao_tnl_pop,37,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:50:36Z,0,4613,3.94,"{'VACUUM END': 3, 'VACUUM START': 3, 'WRITE': 24, 'OPTIMIZE': 6, 'CREATE OR REPLACE TABLE AS SELECT': 1}"
cc_acq,acct_txn,30,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:50:15Z,0,2597,2919.23,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 12, 'CREATE OR REPLACE TABLE AS SELECT': 12}"
cc_acq,acct_txn_stage_1,20,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:49:44Z,0,1042,4769.25,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 7, 'CREATE OR REPLACE TABLE AS SELECT': 7}"
taiwan_savage,propensity_threshold_backtest,11,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:49:02Z,0,222,0.01,"{'VACUUM END': 3, 'VACUUM START': 3, 'CREATE OR REPLACE TABLE AS SELECT': 3, 'OPTIMIZE': 2}"
taiwan_savage,threshold_impacts,9,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:48:35Z,0,111,0.02,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 1, 'CREATE OR REPLACE TABLE AS SELECT': 2}"
cc_acq,txnmob,8,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:50:45Z,0,345,12793.56,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 1, 'CREATE OR REPLACE TABLE AS SELECT': 1}"
cc_acq,app,8,nayanjyoti_sonowal@navyfederal.org,2026-02-16T23:48:43Z,0,128,61.01,"{'VACUUM END': 3, 'VACUUM START': 3, 'OPTIMIZE': 1, 'CREATE OR REPLACE TABLE AS SELECT': 1}"


## Step 8: High Priority Tables - Immediate Vacuum Candidates


In [0]:
if results_df:
    print("\n" + "="*80)
    print("VACUUM COMMANDS FOR HIGH PRIORITY TABLES")
    print("="*80)
    print("\nNote: Default retention is 7 days (168 hours)")
    print("Adjust retention period based on your requirements\n")
    
    high_priority_tables = results_df.filter(col('vacuum_priority') == 'HIGH').collect()
    
    if high_priority_tables:
        for row in high_priority_tables:
            full_table_name = f"{CATALOG}.{row.database}.{row.table}"
            print(f"-- {full_table_name}")
            print(f"-- Versions: {row.version_count}, Potential files to remove: {row.potential_vacuum_files}")
            print(f"VACUUM {full_table_name} RETAIN 168 HOURS;")
            print(f"-- To perform dry run first: VACUUM {full_table_name} RETAIN 168 HOURS DRY RUN;")
            print()
    else:
        print("No high priority tables require immediate vacuum")



VACUUM COMMANDS FOR HIGH PRIORITY TABLES

Note: Default retention is 7 days (168 hours)
Adjust retention period based on your requirements

-- hive_metastore.eats_pipeline.master_escheatment
-- Versions: 277, Potential files to remove: 24423
VACUUM hive_metastore.eats_pipeline.master_escheatment RETAIN 168 HOURS;
-- To perform dry run first: VACUUM hive_metastore.eats_pipeline.master_escheatment RETAIN 168 HOURS DRY RUN;

-- hive_metastore.reporting.blend_loans_bi
-- Versions: 78, Potential files to remove: 292486
VACUUM hive_metastore.reporting.blend_loans_bi RETAIN 168 HOURS;
-- To perform dry run first: VACUUM hive_metastore.reporting.blend_loans_bi RETAIN 168 HOURS DRY RUN;

-- hive_metastore.cc_acq.acctmob
-- Versions: 40, Potential files to remove: 5232
VACUUM hive_metastore.cc_acq.acctmob RETAIN 168 HOURS;
-- To perform dry run first: VACUUM hive_metastore.cc_acq.acctmob RETAIN 168 HOURS DRY RUN;

-- hive_metastore.cc_acq.lao_tnl_pop
-- Versions: 37, Potential files to remove: 

## Step 9: Generate Vacuum Commands


In [0]:
# Read output configuration from widgets
OUTPUT_DATABASE = dbutils.widgets.get("output_database").strip()
OUTPUT_TABLE = dbutils.widgets.get("output_table").strip()

if results_df and OUTPUT_DATABASE and OUTPUT_TABLE:
    try:
        # Add analysis timestamp
        results_with_timestamp = results_df.withColumn(
            'analysis_timestamp', 
            current_timestamp()
        )
        
        # Create full table name with catalog
        full_output_table = f"{CATALOG}.{OUTPUT_DATABASE}.{OUTPUT_TABLE}"
        
        print(f"\nSaving results to: {full_output_table}")
        
        # Write to Delta table
        results_with_timestamp.write.format("delta") \
            .mode("append") \
            .saveAsTable(full_output_table)
        
        print(f"✓ Results saved successfully to {full_output_table}")
        print(f"  - Records saved: {results_with_timestamp.count()}")
        print(f"  - Timestamp: {datetime.now()}")
        
    except Exception as e:
        print(f"✗ Error saving results: {str(e)}")
        print(f"  Attempted to save to: {full_output_table}")
        print(f"  Make sure the database exists and you have write permissions")
        
elif not results_df:
    print("\nNo results to save (no tables with multiple versions found)")
else:
    print("\nOutput database or table name not specified in widgets")



Saving results to: hive_metastore.default.vacuum_analysis_results_v4
✓ Results saved successfully to hive_metastore.default.vacuum_analysis_results_v4
  - Records saved: 159
  - Timestamp: 2026-02-17 18:53:05.495422
